# A424 end-to-end 1D-CNN and temporal Transformer

This experiment trains compact neural networks directly on the saved 424-parcel resting-state fMRI time series. The locked image-only target is AUC **0.622** from the mixed-site RBF-SVM feature baseline. Splits are subject-level and jointly stratified by site and diagnosis. No demographic, QC, motion, or site variables enter either neural network.

In [ ]:
from google.colab import drive
drive.mount('<DRIVE_MOUNT>')
from pathlib import Path
import copy, json, random, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')
ROOT=Path('<DATA_DIR>'); BRAINLM=ROOT/'fmri'/'brainlm_a424'
TS_DIR=BRAINLM/'timeseries_raw'; BENCH=ROOT/'fmri'/'strict_loso_benchmark'
OUT=ROOT/'fmri'/'a424_end_to_end'; OUT.mkdir(parents=True,exist_ok=True)
cohort=pd.read_csv(BENCH/'locked_primary_cohort_409_with_motion.csv',dtype={'subject_id':str})
y=cohort.label.to_numpy('int64'); site=cohort.site.astype(str).to_numpy(); strata=np.char.add(np.char.add(site,'__'),y.astype(str))
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; print('device',DEVICE,'subjects',len(cohort))

## Fixed-length tensor cache
Each subject is linearly resampled to 192 time points. Every parcel is standardized using that subject's own time series, which does not use information from any other subject or fold.

In [ ]:
CACHE=OUT/'a424_timeseries_192.npy'; TARGET_T=192
if CACHE.exists():
    X=np.load(CACHE,mmap_mode='r')
else:
    arr=[]
    target=np.linspace(0,1,TARGET_T)
    for i,sid in enumerate(cohort.subject_id.astype(str),1):
        z=np.asarray(np.load(TS_DIR/f'{sid}.npy'),dtype='float32'); z=np.nan_to_num(z)
        source=np.linspace(0,1,len(z)); r=np.stack([np.interp(target,source,z[:,j]) for j in range(z.shape[1])],axis=1).astype('float32')
        r=(r-r.mean(0,keepdims=True))/(r.std(0,keepdims=True)+1e-5); arr.append(r)
        if i%50==0: print('cached',i,'/',len(cohort),flush=True)
    X=np.stack(arr); np.save(CACHE,X)
print('tensor',X.shape,'size GB',round(X.nbytes/1e9,3))
class FMRIDataset(Dataset):
    def __init__(self,idx,augment=False): self.idx=np.asarray(idx); self.augment=augment
    def __len__(self): return len(self.idx)
    def __getitem__(self,k):
        i=self.idx[k]; z=np.array(X[i],dtype='float32',copy=True)
        if self.augment:
            shift=np.random.randint(-6,7); z=np.roll(z,shift,axis=0)
            if np.random.rand()<.5: z += np.random.normal(0,.015,z.shape).astype('float32')
        return torch.from_numpy(z),torch.tensor(y[i],dtype=torch.float32),int(i)

## Compact neural architectures
The CNN treats parcels as input channels. The Transformer projects the 424-dimensional parcel vector at every time point to a 64-dimensional token and uses two encoder blocks.

In [ ]:
class TemporalCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(
            nn.Conv1d(424,64,9,stride=2,padding=4),nn.BatchNorm1d(64),nn.GELU(),nn.Dropout(.2),
            nn.Conv1d(64,96,7,stride=2,padding=3),nn.BatchNorm1d(96),nn.GELU(),nn.Dropout(.25),
            nn.Conv1d(96,128,5,stride=2,padding=2),nn.BatchNorm1d(128),nn.GELU(),nn.AdaptiveAvgPool1d(1))
        self.head=nn.Sequential(nn.Flatten(),nn.Dropout(.4),nn.Linear(128,1))
    def forward(self,x): return self.head(self.net(x.transpose(1,2))).squeeze(1)
class TemporalTransformer(nn.Module):
    def __init__(self):
        super().__init__(); self.proj=nn.Linear(424,64); self.pos=nn.Parameter(torch.zeros(1,TARGET_T,64))
        layer=nn.TransformerEncoderLayer(64,4,128,dropout=.25,activation='gelu',batch_first=True,norm_first=True)
        self.enc=nn.TransformerEncoder(layer,2); self.norm=nn.LayerNorm(64); self.head=nn.Sequential(nn.Dropout(.4),nn.Linear(64,1))
        nn.init.normal_(self.pos,std=.02)
    def forward(self,x):
        h=self.enc(self.proj(x)+self.pos); return self.head(self.norm(h).mean(1)).squeeze(1)
def seed_all(seed): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
@torch.no_grad()
def predict(model,loader):
    model.eval(); yy=[]; pp=[]; ii=[]
    for xb,yb,ib in loader:
        pp.extend(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy()); yy.extend(yb.numpy()); ii.extend(ib.numpy())
    return np.asarray(yy),np.asarray(pp),np.asarray(ii)
def fit_fold(kind,tr,va,te,seed,max_epochs=25):
    seed_all(seed); model=(TemporalCNN() if kind=='cnn' else TemporalTransformer()).to(DEVICE)
    train=DataLoader(FMRIDataset(tr,True),batch_size=16,shuffle=True,num_workers=0); val=DataLoader(FMRIDataset(va),batch_size=32); test=DataLoader(FMRIDataset(te),batch_size=32)
    pos=(y[tr]==0).sum()/max((y[tr]==1).sum(),1); lossfn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos,device=DEVICE,dtype=torch.float32))
    opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=2e-3); best_auc=-1; best=None; patience=0
    for epoch in range(max_epochs):
        model.train()
        for xb,yb,_ in train:
            opt.zero_grad(); loss=lossfn(model(xb.to(DEVICE)),yb.to(DEVICE)); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        vy,vp,_=predict(model,val); auc=roc_auc_score(vy,vp)
        if auc>best_auc+1e-4: best_auc=auc; best=copy.deepcopy(model.state_dict()); patience=0
        else: patience+=1
        if patience>=6: break
    model.load_state_dict(best); ty,tp,ti=predict(model,test); return ty,tp,ti,best_auc,epoch+1

## Subject-level evaluation
Four mixed-site folds are run for each architecture. Fifteen percent of each outer training fold is reserved for early stopping, jointly stratified by site and label. The test fold is evaluated once.

In [ ]:
rows=[]; preds=[]; outer=StratifiedKFold(n_splits=4,shuffle=True,random_state=2026)
for kind in ['cnn','transformer']:
    for fold,(trainval,te) in enumerate(outer.split(np.zeros(len(y)),strata),1):
        inner=StratifiedShuffleSplit(n_splits=1,test_size=.15,random_state=700+fold); itr,iva=next(inner.split(np.zeros(len(trainval)),strata[trainval]))
        tr=trainval[itr]; va=trainval[iva]; print(kind,'fold',fold,'train/val/test',len(tr),len(va),len(te),flush=True)
        ty,tp,ti,vauc,epochs=fit_fold(kind,tr,va,te,seed=1000+fold)
        pred=(tp>=.5).astype(int); auc=roc_auc_score(ty,tp)
        rows.append({'model':kind,'fold':fold,'n_test':len(te),'auc':auc,'ap':average_precision_score(ty,tp),'balanced_accuracy':balanced_accuracy_score(ty,pred),'best_val_auc':vauc,'epochs':epochs})
        preds.extend({'model':kind,'fold':fold,'subject_id':cohort.subject_id.iloc[i],'site':site[i],'y':int(y[i]),'prob':float(p)} for i,p in zip(ti,tp))
        print('test AUC',round(auc,3),'best val',round(vauc,3),'epochs',epochs,flush=True)
folds=pd.DataFrame(rows); predictions=pd.DataFrame(preds)
summary=(folds.groupby('model').agg(mean_auc=('auc','mean'),sd_auc=('auc','std'),mean_ap=('ap','mean'),mean_balanced_accuracy=('balanced_accuracy','mean')).reset_index().sort_values('mean_auc',ascending=False))
folds.to_csv(OUT/'end_to_end_fold_metrics.csv',index=False); predictions.to_csv(OUT/'end_to_end_predictions.csv',index=False); summary.to_csv(OUT/'end_to_end_summary.csv',index=False)
display(folds.round(3)); display(summary.round(3)); print('Locked image-only baseline AUC: 0.622')
ax=summary.sort_values('mean_auc').plot.barh(x='model',y='mean_auc',xerr='sd_auc',legend=False,figsize=(7,3),color='#4C78A8'); ax.axvline(.622,color='#E45756',ls='--',label='locked baseline 0.622'); ax.set_xlim(.35,.8); ax.legend(); plt.tight_layout(); plt.savefig(OUT/'end_to_end_comparison.png',dpi=180)

## Decision rule
The end-to-end branch passes only if its mean four-fold AUC exceeds 0.622. A single favorable fold is not sufficient. If neither network passes, the result should be retained as a documented negative deep-learning experiment rather than tuned against the test folds.